# Yield Curve Regimes & Equity Positioning
## Week 1: Build the dataset

**Research question:** How have U.S. equity sectors performed in each yield-curve regime since 1976, and what does today's regime suggest for sector positioning?

This notebook downloads every series the project needs, puts it on one monthly timeline, checks it against events you already know, and saves a clean dataset for Weeks 2–6.

**How to run it:** open it in [Google Colab](https://colab.research.google.com) (File → Upload notebook), then choose **Runtime → Run all**. It takes under a minute and needs no installs or API keys.

| Source | Series | Why we need it |
|---|---|---|
| FRED (St. Louis Fed) | 10-year, 2-year and 3-month Treasury yields, daily | Level and slope of the yield curve |
| FRED | Effective fed funds rate, monthly | Context for what the Fed was doing in each regime |
| FRED | NBER recession indicator, monthly | Shading charts and checking the curve's track record |
| Ken French Data Library | 12 industry portfolios, monthly returns | Sector performance back to 1926 |
| Ken French Data Library | Market return and 1-month T-bill rate | Benchmark for "did this sector beat the market?" |

### 1. Setup
The 2-year Treasury series starts in June 1976, so that is where the sample begins.

In [ ]:
import io
import zipfile
from pathlib import Path

import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import HTML, display

START = "1976-06-01"          # first month with a 2-year Treasury yield
DATA = Path("data");    DATA.mkdir(exist_ok=True)
FIGS = Path("figures"); FIGS.mkdir(exist_ok=True)

FRED_URL   = "https://fred.stlouisfed.org/graph/fredgraph.csv?id={}"
FRENCH_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/{}"
HEADERS    = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
                            "(KHTML, like Gecko) Chrome/126.0 Safari/537.36"}
TIMEOUT    = (10, 20)         # seconds to connect, seconds to wait for data

def files_here(*patterns):
    """Files already in Colab's working folder (uploads land here), as {name: bytes}."""
    return {p.name: p.read_bytes() for pat in patterns for p in Path(".").glob(pat) if p.is_file()}

def show_links(title, links):
    """Show clickable download links that open in a new browser tab."""
    items = "".join(f'<li><a href="{url}" target="_blank">{label}</a></li>' for label, url in links)
    display(HTML(f"<p><b>{title}</b></p><ol>{items}</ol>"))

def ask_for_files(what):
    """Show Colab's upload button and return {file name: bytes} for the files you pick."""
    try:
        from google.colab import files
    except ImportError:
        raise RuntimeError(f"Put the {what} files in this notebook's folder, then run this cell again.")
    print(f"After downloading, click 'Choose Files' below and select the {what} files from your Downloads folder.")
    return files.upload()

print("pandas", pd.__version__)

### 2. Download Treasury yields, fed funds and recessions from FRED
FRED serves each series as a CSV, so no account or API key is needed. Missing days (holidays) come through blank and are dropped later.

**If FRED doesn't answer Colab** (it sometimes blocks cloud servers), this cell switches to Plan B within about 20 seconds: it shows five download links and an upload button. Click each link to download the file in your own browser, then click **Choose Files** and select all five. Files are recognized by their contents, so their names don't matter. Once uploaded, re-running this cell uses them instantly.

In [ ]:
FRED_SERIES = {
    "DGS10":    "10-year Treasury yield, daily (%)",
    "DGS2":     "2-year Treasury yield, daily (%)",
    "DGS3MO":   "3-month Treasury bill yield, daily (%)",
    "FEDFUNDS": "Effective federal funds rate, monthly average (%)",
    "USREC":    "NBER recession indicator, monthly (1 = recession)",
}

def parse_fred(text):
    """Read one FRED CSV (as text) into (series id, numeric Series indexed by date)."""
    df = pd.read_csv(io.StringIO(text.lstrip("﻿")), na_values=[".", ""])
    sid = str(df.columns[1]).strip()
    df.columns = ["date", sid]                     # FRED has renamed its date column before
    df["date"] = pd.to_datetime(df["date"])
    return sid, pd.to_numeric(df.set_index("date")[sid], errors="coerce").loc[START:]

def fred_from_files(files):
    """Find FRED series among {name: bytes}, identified by the column header, not the file name."""
    found = {}
    for name, raw in files.items():
        text = raw.decode("utf-8", errors="ignore").lstrip("﻿")
        header = [c.strip().strip('"') for c in text.split("\n", 1)[0].split(",")]
        if len(header) == 2 and header[1] in FRED_SERIES:
            sid, series = parse_fred(text)
            found[sid] = series
    return found

# 1) Use files you've already uploaded, so re-running this cell is instant.
fred = fred_from_files(files_here("*.csv", "*.CSV"))

# 2) Otherwise ask FRED directly, giving up after about 20 seconds if it doesn't answer.
missing = [sid for sid in FRED_SERIES if sid not in fred]
try:
    for sid in missing:
        r = requests.get(FRED_URL.format(sid), headers=HEADERS, timeout=TIMEOUT)
        r.raise_for_status()
        fred[sid] = parse_fred(r.text)[1]
except requests.RequestException as e:
    print(f"FRED didn't answer Colab ({type(e).__name__}). It sometimes blocks cloud servers, so switching to Plan B.\n")

# 3) Plan B: download in your own browser, then upload here.
missing = [sid for sid in FRED_SERIES if sid not in fred]
if missing:
    show_links("Plan B: click each link to download the file. If one opens as text instead, "
               "press Ctrl+S (Windows) or Cmd+S (Mac) to save it.",
               [(f"{sid}: {FRED_SERIES[sid]}", FRED_URL.format(sid)) for sid in missing])
    fred.update(fred_from_files(ask_for_files(f"{len(missing)} FRED")))

missing = [sid for sid in FRED_SERIES if sid not in fred]
if missing:
    raise RuntimeError(f"Still missing {', '.join(missing)}. Download those links, then run this cell again.")

for sid, s in fred.items():
    s = s.dropna()
    print(f"{sid:9s} {s.index.min():%Y-%m-%d} to {s.index.max():%Y-%m-%d}   {len(s):>6,} obs   {FRED_SERIES[sid]}")

### 3. Put the yields on a monthly timeline
We keep the **last trading day of each month**, not the monthly average. The closing yield on that day is exactly what an investor knew at month-end. A monthly average is also known by then, but it lags turning points, so regimes would appear to start later than they really did. Week 3 depends on knowing exactly what was known when.

Spreads are in percentage points. A negative 10y–2y spread means the curve is **inverted**.

In [ ]:
def monthly(s, how="last"):
    """One value per calendar month: the last valid observation (or the mean)."""
    g = s.dropna().groupby(s.dropna().index.to_period("M"))
    return g.last() if how == "last" else g.mean()

curve = pd.DataFrame({
    "y10": monthly(fred["DGS10"]),
    "y2":  monthly(fred["DGS2"]),
    "y3m": monthly(fred["DGS3MO"]),
})
curve["spread_10y2y"] = curve["y10"] - curve["y2"]
curve["spread_10y3m"] = curve["y10"] - curve["y3m"]

macro = pd.DataFrame({
    "fed_funds": monthly(fred["FEDFUNDS"]),
    "recession": monthly(fred["USREC"]),
})

curve.tail()

### 4. Download sector and market returns from the Ken French Data Library
Professor Kenneth French (Dartmouth) publishes monthly returns for U.S. stocks grouped into 12 industries, going back to 1926. Each file holds several tables; the first is **value-weighted monthly returns**, which is the one we want. Returns arrive in percent and are converted to decimals (1.45 → 0.0145).

| Code | Name used here | What's in it |
|---|---|---|
| NoDur | Nondurables | Food, tobacco, textiles, apparel, toys |
| Durbl | Durables | Cars, TVs, furniture, household appliances |
| Manuf | Manufacturing | Machinery, trucks, planes, office furniture, paper |
| Enrgy | Energy | Oil, gas and coal extraction and products |
| Chems | Chemicals | Chemicals and allied products |
| BusEq | Tech | Computers, software and electronic equipment |
| Telcm | Telecom | Telephone and television transmission |
| Utils | Utilities | Utilities |
| Shops | Retail | Wholesale, retail and some services |
| Hlth | Healthcare | Healthcare, medical equipment and drugs |
| Money | Finance | Banks, insurance and other finance |
| Other | Other | Mines, construction, transport, hotels, entertainment and the rest |

If Dartmouth's site doesn't answer Colab either, this cell uses the same Plan B: two download links (zip files) and an upload button. You can upload the zips as they are.

In [ ]:
FRENCH_FILES = {
    "sectors": ("12_Industry_Portfolios_CSV.zip", "12 industry portfolios (zip)"),
    "factors": ("F-F_Research_Data_Factors_CSV.zip", "Fama-French factors with the market return (zip)"),
}

def french_text(raw):
    """Bytes of a Ken French .zip or .csv file -> the CSV text inside."""
    if raw[:2] == b"PK":                            # zip files start with "PK"
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            raw = z.read(z.namelist()[0])
    return raw.decode("latin-1")

def parse_french(text):
    """Return the first monthly table in a Ken French file, in decimals."""
    lines = text.splitlines()

    def is_month(line):
        first = line.split(",")[0].strip()
        return len(first) == 6 and first.isdigit()

    # The monthly table starts at a header row beginning with "," followed by a YYYYMM row.
    start = next((i for i in range(len(lines) - 1)
                  if lines[i].startswith(",") and is_month(lines[i + 1])), None)
    if start is None:
        raise ValueError("No monthly table found")
    header = [c.strip() for c in lines[start].split(",")[1:]]

    rows = []
    for line in lines[start + 1:]:
        if not is_month(line):
            break                                   # the table ends at the first non-YYYYMM row
        rows.append([p.strip() for p in line.split(",")])

    months = pd.to_datetime([r[0] for r in rows], format="%Y%m").to_period("M")
    df = pd.DataFrame([r[1:] for r in rows], index=months, columns=header).astype(float)
    return df.mask(df <= -99.99) / 100               # -99.99 and -999 mean "missing"

SECTOR_NAMES = {
    "NoDur": "Nondurables", "Durbl": "Durables", "Manuf": "Manufacturing", "Enrgy": "Energy",
    "Chems": "Chemicals", "BusEq": "Tech", "Telcm": "Telecom", "Utils": "Utilities",
    "Shops": "Retail", "Hlth": "Healthcare", "Money": "Finance", "Other": "Other",
}

def french_from_files(files):
    """Find the two Ken French tables among {name: bytes}, identified by their column headers."""
    found = {}
    for name, raw in files.items():
        try:
            text = french_text(raw)
            head = text[:5000]
            kind = "sectors" if "NoDur" in head else "factors" if "Mkt-RF" in head else None
            if kind:
                found[kind] = parse_french(text)
        except (zipfile.BadZipFile, ValueError):
            continue
    return found

# 1) Files you've already uploaded  2) Dartmouth directly  3) Plan B: download and upload.
french = french_from_files(files_here("*.zip", "*.csv", "*.CSV"))
try:
    for kind, (zip_name, _) in FRENCH_FILES.items():
        if kind not in french:
            r = requests.get(FRENCH_URL.format(zip_name), headers=HEADERS, timeout=TIMEOUT)
            r.raise_for_status()
            french[kind] = parse_french(french_text(r.content))
except requests.RequestException as e:
    print(f"Dartmouth's site didn't answer Colab ({type(e).__name__}), so switching to Plan B.\n")

missing = [kind for kind in FRENCH_FILES if kind not in french]
if missing:
    show_links("Plan B: click each link to download the zip file:",
               [(FRENCH_FILES[k][1], FRENCH_URL.format(FRENCH_FILES[k][0])) for k in missing])
    french.update(french_from_files(ask_for_files(f"{len(missing)} Ken French")))
missing = [kind for kind in FRENCH_FILES if kind not in french]
if missing:
    raise RuntimeError("Still missing the Ken French files. Download the links above, then run this cell again.")

sectors = french["sectors"].rename(columns=SECTOR_NAMES)
factors = french["factors"].copy()
factors["Market"] = factors["Mkt-RF"] + factors["RF"]     # total market return = excess return + T-bill

print("Sectors:", sectors.index.min(), "to", sectors.index.max(), "|", ", ".join(sectors.columns))
print("Market: ", factors.index.min(), "to", factors.index.max())

### 5. Combine everything into one monthly table
Each row is one month. The Ken French library usually runs a month or two behind FRED, so the newest months may show yields with blank returns. That's expected.

In [ ]:
SECTORS = list(SECTOR_NAMES.values())

data = (curve
        .join(macro, how="left")
        .join(factors[["Market", "RF"]], how="left")
        .join(sectors[SECTORS], how="left"))
data = data[data["y10"].notna() & data["y2"].notna()]
data.index.name = "month"

print(f"{len(data)} months, {data.index.min()} to {data.index.max()}")
print(f"Returns available through {data['Market'].last_valid_index()}")
data.tail(3)

### 6. Sanity checks against events you already know
Good research checks its data before trusting it. Each printout below should match something you can verify:
- The worst market months should include **October 1987** (Black Monday) and **October 2008** (the financial crisis).
- The curve should show **inversions** around 1978–80, 1989, 2000, 2006–07, 2019 and a long one starting in 2022.
- Missing values should only appear where a series hasn't started yet (the 3-month bill before 1981) or hasn't been published yet.

In [ ]:
def episodes(flags):
    """Turn a True/False monthly series into (first month, last month) runs of True."""
    runs, start, prev = [], None, None
    for month, flag in flags.items():
        if flag and start is None:
            start = month
        if not flag and start is not None:
            runs.append((start, prev)); start = None
        prev = month
    if start is not None:
        runs.append((start, prev))
    return runs

print("Missing values per column:")
print(data.isna().sum()[lambda s: s > 0].to_string() or "  none")

print("\nWorst five market months (total return):")
print((data["Market"].nsmallest(5) * 100).round(1).astype(str).add("%").to_string())

print("\nMonths with an inverted curve (10y minus 2y below zero):")
for a, b in episodes(data["spread_10y2y"] < 0):
    print(f"  {a} to {b}   {(b - a).n + 1:>2} months   deepest {data.loc[a:b, 'spread_10y2y'].min():+.2f} pts")

print("\nNBER recessions in the sample:")
for a, b in episodes(data["recession"] == 1):
    print(f"  {a} to {b}   {(b - a).n + 1:>2} months")

### 7. First chart: the yield curve spread since 1976
Two common ways to measure the curve's slope. The **10y–2y** spread is the one most often quoted in markets. The **10y–3m** spread is the one the New York Fed uses in its recession probability model. Shaded bands are NBER recessions.

In [ ]:
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
BLUE, ORANGE, RECESSION = "#2a78d6", "#eb6834", "#d6d5d0"

t = data.index.to_timestamp()
fig, ax = plt.subplots(figsize=(11, 4.6), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")

ymin = min(data["spread_10y2y"].min(), data["spread_10y3m"].min()) - 0.3
ymax = max(data["spread_10y2y"].max(), data["spread_10y3m"].max()) + 0.3
ax.fill_between(t, ymin, ymax, where=(data["recession"] == 1).to_numpy(),
                color=RECESSION, alpha=0.6, linewidth=0, step="mid", label="NBER recession")
ax.axhline(0, color=MUTED, linewidth=1)
ax.plot(t, data["spread_10y3m"], color=ORANGE, linewidth=1.4, label="10-year minus 3-month")
ax.plot(t, data["spread_10y2y"], color=BLUE, linewidth=1.8, label="10-year minus 2-year")

ax.set_ylim(ymin, ymax)
ax.set_title("U.S. Treasury yield curve spread, monthly since 1976", loc="left", color=INK, fontsize=13)
ax.set_ylabel("Percentage points (below 0 = inverted)", color=MUTED)
ax.tick_params(colors=MUTED)
ax.grid(axis="y", color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(GRID)
ax.legend(loc="lower right", bbox_to_anchor=(1, 1.0), frameon=False, labelcolor=INK, ncol=3, fontsize=9)
fig.text(0.01, 0.01, "Sources: FRED (Federal Reserve Bank of St. Louis); NBER recession dates via FRED",
         color=MUTED, fontsize=8)
fig.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig(FIGS / "yield_curve_spread.png", dpi=200, facecolor=fig.get_facecolor())
plt.show()

### 8. Save the dataset
This writes `data/monthly_dataset.csv` (and the chart) to the Colab session. Colab deletes session files when it disconnects, so the next cell downloads both to your computer. Upload the CSV to your GitHub repo too, so Week 3 can start from it.

In [ ]:
data.round(4).to_csv(DATA / "monthly_dataset.csv")   # 4 decimals = full precision of the source data
print("Saved", DATA / "monthly_dataset.csv", "-", data.shape[0], "rows x", data.shape[1], "columns")

try:                                   # only works inside Google Colab
    from google.colab import files
    files.download(str(DATA / "monthly_dataset.csv"))
    files.download(str(FIGS / "yield_curve_spread.png"))
except ImportError:
    print("Not running in Colab: find the files in the data/ and figures/ folders.")

### 9. Week 1 study questions
Write your answers in a doc. They become the background section of your research note, and they are the questions an asset management interviewer is likely to ask.

1. In one or two sentences each: what does the 10y–2y spread measure, and why does the 2-year yield move so closely with expectations for Fed policy?
2. From the sanity-check printout, list each inversion. Which ones were followed by an NBER recession, and how many months later did it start?
3. What has happened since the inversion that began in 2022? What does that tell you about the curve as a *timing* signal versus a *warning* signal?
4. Why does this notebook use the last trading day of each month instead of the monthly average?
5. The NBER announces recessions months after they start. Why is the recession indicator fine for shading a chart but dangerous to use as a trading signal?

**Next (Week 2):** read up on what drives each of the four regimes (bull steepener, bear steepener, bull flattener, bear flattener) and write one paragraph on each before we classify months in Week 3.